### Import Libraries

In [ ]:
import os
import shutil
from huggingface_hub import hf_hub_download
import zipfile
import pandas as pd
from pathlib import Path
import ast

/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initialization

In [3]:
repo_id = "AI4Patents/IMPACT"

# Remote dir
# dataset_dir = Path("/home/Keith/[project] patent_research/impact_dataset")

# Local dir
dataset_dir = Path("impact_dataset")

years = range(2022, 2022 + 1)

### Download Dataset

In [5]:
os.makedirs(dataset_dir, exist_ok=True)

In [6]:
def download_file(filename: str):
    print(f"Downloading {filename} ...")

    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type="dataset",
        local_dir=dataset_dir,
        local_dir_use_symlinks=False,
        resume_download=True
    )

    print(f"Saved to {path}")

In [16]:
for year in years:
    csv_name = f"{year}.csv"
    zip_name = f"{year}.zip"

    download_file(csv_name)
    download_file(zip_name)

/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/home/yishin/miniconda3/envs/patent/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Saved to impact_dataset/2022.csv
Saved to impact_dataset/2022.zip


### Unzip Dataset

In [17]:
def delete_zip_by_name(zip_name: str, data_dir=dataset_dir):
    data_dir = Path(data_dir)

    target = zip_name if zip_name.lower().endswith(".zip") else f"{zip_name}.zip"
    zip_path = data_dir / target

    if not zip_path.exists():
        raise FileNotFoundError(f"Zip not found: {zip_path}")

    zip_path.unlink()
    print(f"Deleted: {zip_path.name}")
    return


In [18]:
def remove_macosx(folder_name: str, data_dir: Path):
    target_dir = data_dir / folder_name
    temp_dir = data_dir / "temp"

    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)

    kept = None

    for child in target_dir.iterdir():
        if child.is_dir() and child.name == folder_name:
            kept = child
        else:
            shutil.rmtree(child) if child.is_dir() else child.unlink()

    if kept is None:
        raise FileNotFoundError(f"Did not find '{folder_name}' inside {target_dir}")

    moved = temp_dir / kept.name
    kept.rename(moved)

    shutil.rmtree(target_dir)
    moved.rename(target_dir)
    shutil.rmtree(temp_dir)

In [19]:
def unzip_all(data_dir=dataset_dir):
    zip_files = [f for f in os.listdir(data_dir) if f.endswith(".zip")]

    if not zip_files:
        print("No zip files found.")
        return

    for zfile in zip_files:
        zip_path = os.path.join(data_dir, zfile)

        if not os.path.isfile(zip_path):
            print(zip_path)
            print(f"[ERROR] File not found, skipping: {zip_path}")
            continue

        folder_name = zfile.replace(".zip", "")
        extract_dir = os.path.join(data_dir, folder_name)

        if os.path.exists(extract_dir):
            print(f"[SKIPPED] {zfile} already extracted.")
            continue

        print(f"[UNZIPPING] {zfile} → {extract_dir}")

        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

        print(f"[DONE] Extracted to: {extract_dir}")

        remove_macosx(folder_name, dataset_dir)
        delete_zip_by_name(folder_name, dataset_dir)


    print("\nAll .zip files processed.")

In [21]:
unzip_all()

[UNZIPPING] 2022.zip → impact_dataset/2022
[DONE] Extracted to: impact_dataset/2022
Deleted: 2022.zip

All .zip files processed.


## Dataframe Management

In [ ]:
list_cols = ["class_search", "file_names", "fig_desc"]
text_cols = ["title", "claim", "caption"]

In [ ]:
def find_img_path(sample, data_dir=dataset_dir):
    year = sample["year"]
    sample_file_names = sample["file_names"]

    if not isinstance(sample_file_names, (list, tuple)) or len(sample_file_names) == 0:
        return []

    first_name = sample_file_names[0]

    if not isinstance(first_name, str):
        return []

    parts = first_name.split("-")
    if len(parts) >= 2:
        folder_name = "-".join(parts[:2])
    else:
        folder_name = parts[0]

    # 4. Build folder path (cast year to str for safety)
    folder_path = os.path.join(data_dir, str(year), folder_name)

    img_paths = [
        os.path.join(folder_path, fn)
        for fn in sample_file_names
        if isinstance(fn, str)
    ]

    return img_paths


In [ ]:
def safe_literal_eval(x):
    if isinstance(x, (list, tuple)):
        return list(x)
    if not isinstance(x, str) or x.strip() == "":
        return []
    try:
        v = ast.literal_eval(x)
    except Exception:
        return []
    
    return list(v) if isinstance(v, (list, tuple)) else []

In [ ]:
def merge_all_csv(
    data_dir=dataset_dir, 
    source_file="year", 
    list_cols=None,
    year_list: list = []
):
    csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

    if year_list == []:
        filtered_csv = csv_files
    else:
        filtered_csv = [f for f in csv_files if f.replace(".csv", "") in year_list]

    if not filtered_csv:
        print("No csv files found.")
        return

    frames = []
    for csv in filtered_csv:
        csv_path = os.path.join(data_dir, csv)
        df = pd.read_csv(csv_path)

        for col in list_cols:
            if col in df.columns:
                df[col] = df[col].apply(
                    lambda x: safe_literal_eval(x)
                    if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
                    else x
                )
        
        filename = os.path.basename(csv_path).split('.')[0]

        df[source_file] = filename
        cols = [source_file] + [c for c in df.columns if c != source_file]
        df = df[cols]

        frames.append(df)

    merged = pd.concat(frames, ignore_index=True)

    merged["file_names"] = merged.apply(
        lambda row: find_img_path(row, data_dir=data_dir),
        axis=1,
    )
    
    return merged

        

In [ ]:
# Impact_df = merge_all_csv(list_cols=list_cols, year_list=year_list)
Impact_df = merge_all_csv(list_cols=list_cols)

In [ ]:
cols = ["year", "date", "title", "caption", "file_names", "fig_desc", "class", "class_search"]

Impact_df = Impact_df[cols]

In [ ]:
Impact_df.head()

,year,date,title,caption,file_names,fig_desc,class,class_search
0,2021,20211207,Display screen or portion thereof with a graph...,"The image is a square, and it displays a grap...",[impact_dataset/2021/USD0937859-20211207/USD09...,[The FIGURE is a from view of a display screen...,D14486,"[1404, D14486, D14486, D14486, D14485, D14492,..."
1,2021,20210126,Garment with a side pocket,The image is a square-shaped illustration of ...,[impact_dataset/2021/USD0908314-20210126/USD09...,[FIG. 1 is a front left perspective view of th...,"D 2728, D2840","[0202, D 2728, D 2839, D 2829, D 2750, D 2839,..."
2,2021,20210406,Quilted fabric,"The image is a square shape, and its function...",[impact_dataset/2021/USD0915081-20210406/USD09...,[A portion of the disclosure of this patent do...,D 5 59,"[0505, D 5 59, D 5 99, D 5 53, D 5 62, D 5 63,..."
3,2021,20210316,Display screen or portion thereof with animate...,The image is a square-shaped display screen w...,[impact_dataset/2021/USD0913312-20210316/USD09...,[FIG. 1 is a front view of a display screen or...,"D14486,D14487","[1404, D14486, D14486, D14485, D14486, D14488,..."
4,2021,20210907,Bat,"The image is a long, thin, and elongated shap...",[impact_dataset/2021/USD0930094-20210907/USD09...,[FIG. 1 is a perspective view of a bat showing...,D21725,"[2102, D21725, 473568, D21725, D21725, 473568,..."


In [ ]:
Impact_df.tail()

,year,date,title,caption,file_names,fig_desc,class,class_search
99013,2022,20220614,Socket,"The image is a white drawing of a socket, whi...",[impact_dataset\2022\USD0954652-20220614\USD09...,[FIG. 1 is a front elevational view of a socke...,"D131374,D131372","[1303, D131374, D131372, D131398, D13110, D131..."
99014,2022,20220125,Mobile payment terminal,The image is a square-shaped drawing of a cel...,[impact_dataset\2022\USD0941911-20220125\USD09...,[FIG. 1 is a front perspective view of a mobil...,D18 46,"[1801, D18 46, D14387, D142037, D14371, 34517..."
99015,2022,20220322,Watch band storage case,"The image is a square shape, and it shows a w...",[impact_dataset\2022\USD0946280-20220322\USD09...,[FIG. 1 is a front view of the watch band stor...,"D 3301, D3903","[0301, D 3301, D 3319, 150106, 132315, D 3205,..."
99016,2022,20221129,Sign stake,"The image is a long, thin, and rectangular sh...",[impact_dataset\2022\USD0971009-20221129\USD09...,[FIG. 1 is a front perspective view of the sig...,"D 8367, D8370","[0805, D 8367, D 8356, 47 47, 2482182, D 8370..."
99017,2022,20220802,Stylus,"The image is a white drawing of a stylus, whi...",[impact_dataset\2022\USD0959433-20220802\USD09...,[FIG. 1 is a perspective view of a stylus show...,D14411,"[1402, D14411, D14411, D14411, D14411, D14411,..."


In [ ]:
Impact_df.shape[0]

99018

### Save Impact DF

In [ ]:
Impact_df.to_csv("Impact.csv", index=False, encoding="utf-8")

### Load Impact DF

In [1]:
import ast
import pandas as pd

In [2]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [4]:
Impact_df = pd.read_csv("Impact_2022_Sub_KW.csv",encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,file_names,Loc_class,main_class,sub_class,llava_output,components
0,Article of footwear,[impact_dataset/2022/USD0943877-20220222/USD09...,{02-04},2,4,The image you've provided appears to be a tech...,"sole, heel, toe, upper, tongue, laces, eyelets"
1,Shoe,[impact_dataset/2022/USD0950931-20220510/USD09...,{02-04},2,4,"The image shows a pair of shoes, and I will de...","sole, heel, upper, tongue, laces, eyelets"
2,Rear combination lamp for automobile,[impact_dataset/2022/USD0957706-20220712/USD09...,"{02-07, 26-06}",2,7,The image you've provided appears to be a gray...,"base, housing, lens, switch, cord, bulb, refle..."
3,Portable light beacon,[impact_dataset/2022/USD0959716-20220802/USD09...,"{02-07, 26-02}",2,7,The image you've provided appears to be a tech...,"base, stem, head, lens, switch, battery compar..."
4,Water shoe,[impact_dataset/2022/USD0960535-20220816/USD09...,{02-04},2,4,The image you've provided appears to be a line...,"sole, upper, tongue, laces, heel, toe cap"
